In [63]:
import re
import ast
import pandas as pd
from scipy.stats import ttest_rel
from itertools import combinations
import numpy as np

## For total acccuracy

In [65]:
def parse_run_string(s):
    # From CHAT: Replace np.float64(0.123) -> 0.123
    s = re.sub(r'np\.float64\(([^()]*)\)', r'\1', s)
    return ast.literal_eval(s)

def extract_final_accuracy(s):
    data = parse_run_string(s)
    return float(data[-1][1])

def extract_auc(s):
    data = parse_run_string(s)
    ys = [float(acc) for _, acc in data]
    return sum(ys) / len(ys)

In [66]:
def clean_total_data(df):
    df_clean = pd.DataFrame({
        "random": df["random"].apply(extract_final_accuracy),
        "margin": df["margin"].apply(extract_final_accuracy),
        "least_conf": df["least_confident"].apply(extract_final_accuracy),
        "entropy": df["entropy"].apply(extract_final_accuracy),
    })
    return df_clean

In [67]:
df_et = clean_total_data(pd.read_csv("results/even_total_accuracy.csv"))
df_ut = clean_total_data(pd.read_csv("results/uneven_total_accuracy.csv"))

In [68]:
df_et.head()

,random,margin,least_conf,entropy
0,0.451638,0.417577,0.278471,0.258710
1,0.457358,0.433177,0.313313,0.297192
2,0.437337,0.426417,0.312793,0.269371
3,0.457098,0.404836,0.311752,0.329173
4,0.428497,0.427457,0.277951,0.291212


In [86]:
df_et_means = pd.DataFrame({
            "random": np.mean(df_et["random"]),
            "margin": np.mean(df_et["margin"]),
            "least_conf": np.mean(df_et["least_conf"]),
            "entropy": np.mean(df_et["entropy"]),
        },index=[0])

df_ut_means = pd.DataFrame({
            "random": np.mean(df_ut["random"]),
            "margin": np.mean(df_ut["margin"]),
            "least_conf": np.mean(df_ut["least_conf"]),
            "entropy": np.mean(df_ut["entropy"]),
        },index=[0])

In [87]:
print("Even")
df_et_means

Even


,random,margin,least_conf,entropy
0,0.431992,0.41702,0.305632,0.281139


In [85]:
print("Uneven")
df_ut_means

Uneven


,random,margin,least_conf,entropy
0,0.61883,0.623083,0.540177,0.52976


## For class accuracy

In [70]:
def extract_class_final_accuracy(s):
    data = parse_run_string(s)  # list of class curves
    
    return [float(class_curve[-1][1]) for class_curve in data]

def extract_class_auc(s):
    data = parse_run_string(s)
    
    return [
        sum(float(acc) for _, acc in class_curve) / len(class_curve)
        for class_curve in data
    ]

In [71]:
def clean_class_data(df):
    rows = []

    for run_idx in range(len(df)):
        row = df.iloc[run_idx]

        random = extract_class_final_accuracy(row["random"])
        margin = extract_class_final_accuracy(row["margin"])
        least = extract_class_final_accuracy(row["least_confident"])
        entropy = extract_class_final_accuracy(row["entropy"])

        n_classes = len(random)

        for c in range(n_classes):
            rows.append({
                "run": run_idx,
                "class": c,
                "random": random[c],
                "margin": margin[c],
                "least_conf": least[c],
                "entropy": entropy[c],
            })

    return pd.DataFrame(rows)

In [72]:
df_ec = clean_class_data(pd.read_csv("results/even_class_accuracy.csv"))
df_uc = clean_class_data(pd.read_csv("results/uneven_class_accuracy.csv"))

In [73]:
df_ec.head(10)

,run,class,random,margin,least_conf,entropy
0,0,0,0.087432,0.571949,0.446266,0.132969
1,0,1,0.522769,0.167577,0.522769,0.459016
2,0,2,0.420765,0.406193,0.504554,0.193078
3,0,3,0.630909,0.801818,0.000000,0.000000
4,0,4,0.460000,0.576364,0.120000,0.463636
5,0,5,0.076503,0.134791,0.296903,0.551913
6,0,6,0.961818,0.263636,0.060000,0.010909
7,1,0,0.571949,0.409836,0.147541,0.120219
8,1,1,0.283636,0.203636,0.547273,0.581818
9,1,2,0.385455,0.323636,0.721818,0.809091


## Statistical tests

In [ ]:
def paired_tests_total(df, alpha=0.05):
    methods = df.columns
    results = []

    for m1, m2 in combinations(methods, 2):
        t, p = ttest_rel(df[m1], df[m2])
        results.append((m1, m2, t, p))

    # Bonferroni correction
    alpha_corr = alpha / len(results)
    
    print(f"Bonferroni corrected alpha: {alpha_corr:.5f}\n")

    for m1, m2, t, p in results:
        sig = "SIGNIFICANT" if p < alpha_corr else "ns"
        print(f"{m1} vs {m2}: t={t:.3f}, p={p:.5f} → {sig}")
    print()

In [ ]:
paired_tests_total(df_et)
paired_tests_total(df_ut)

Bonferroni corrected alpha: 0.00833

random vs margin: t=2.865, p=0.00612 → SIGNIFICANT
random vs least_conf: t=23.083, p=0.00000 → SIGNIFICANT
random vs entropy: t=28.131, p=0.00000 → SIGNIFICANT
margin vs least_conf: t=16.927, p=0.00000 → SIGNIFICANT
margin vs entropy: t=24.732, p=0.00000 → SIGNIFICANT
least_conf vs entropy: t=4.162, p=0.00013 → SIGNIFICANT

Bonferroni corrected alpha: 0.00833

random vs margin: t=-0.624, p=0.53558 → ns
random vs least_conf: t=9.806, p=0.00000 → SIGNIFICANT
random vs entropy: t=11.528, p=0.00000 → SIGNIFICANT
margin vs least_conf: t=9.622, p=0.00000 → SIGNIFICANT
margin vs entropy: t=11.311, p=0.00000 → SIGNIFICANT
least_conf vs entropy: t=1.368, p=0.17754 → ns

